# DS5 Step 2 Path B — Intervention-Sensitive Phenotyping

**Notebook:** `105_ds5_intervention_sensitive_phenotype.ipynb`  
**Stage:** Stage 4, Step 2 — Path B  
**Date:** 2026-04-20

Tests whether individual differences in **friction-response diversity** (`H_out_resid`) form stable phenotype types, and how these relate to the existing DS5 phenotype taxonomy.

**Design (condition-aware):**
- Reintervention = PRIMARY clustering condition
- Control = counterfactual baseline with clustering attempt
- Default = descriptive only (no clustering, consistent with Path E Outcome D)

**Blueprint:** `outputs/reports/stage4/ds5_path_b_intervention_sensitive_blueprint.md`

In [ ]:
# [0] Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                              davies_bouldin_score, adjusted_rand_score)
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency, pearsonr
from pathlib import Path

RANDOM_STATE   = 42
N_BOOTSTRAPS   = 100
SAMPLE_FRAC    = 0.80
MIN_CELL       = 30
JACCARD_STABLE = 0.75
JACCARD_MARG   = 0.60

# Intended features (before correlation gate)
INTENDED_FEATURES = ['H_out_resid', 'x_mean', 'sig_hat']
CONDITIONS = ['control', 'default', 'reintervention']

COND_COLORS   = {'control': '#2196F3', 'default': '#FF9800', 'reintervention': '#4CAF50'}
COND_MARKERS  = {'control': 'o', 'default': 's', 'reintervention': '^'}
CLUSTER_COLORS_B = {'B_fr1': '#9C27B0', 'B_fr2': '#FF5722', 'B_fr3': '#009688'}
CELL_COLORS = {'B_behav1': '#3F51B5', 'B_behav2': '#E91E63', 'B_behav3': '#FF9800', 'A_all': '#607D8B'}

ROOT      = Path('.')
PHENO_PATH = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
FIG_PATH   = PHENO_PATH / 'figures'
FIG_PATH.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

In [ ]:
# [1] Load data
feat   = pd.read_csv(PHENO_PATH / 'ds5_phenotype_feature_matrix.csv')
path_a = pd.read_csv(PHENO_PATH / 'ds5_behav_only_cluster_assignments.csv')
path_f = pd.read_csv(PHENO_PATH / 'ds5_hierarchical_assignments.csv')
path_e = pd.read_csv(PHENO_PATH / 'ds5_condition_cluster_assignments.csv')
survey_path = ROOT / 'outputs' / 'features' / 'ds5' / 'ds5_survey_features.csv'
survey = pd.read_csv(survey_path) if survey_path.exists() else None

print('Feature matrix:', feat.shape)
print('Path A cols:', path_a.columns.tolist())
print('Path F cols:', path_f.columns.tolist())
print('Path E cols:', path_e.columns.tolist())

In [ ]:
# [2] Merge master dataframe (N=414)
PSYCH = ['belong_mean_pre', 'ostracism_mean_pre', 'fomo_mean_pre',
         'digital_wellbeing_pre', 'prob_self_pre', 'social_media_wellbeing_pre']

LOAD_COLS = ['participant_id_canonical', 'study_condition',
             'H_out_resid', 'x_mean', 'sig_hat', 'H_emp',
             'H_out', 'mu_hat', 'sigma_mu_ratio', 'chi_behav', 'H_hour']

df = feat[LOAD_COLS].copy()
df = df.merge(path_a[['participant_id_canonical', 'cluster_behav']],
              on='participant_id_canonical', how='left')
df = df.merge(path_f[['participant_id_canonical', 'profile_p1', 'cell_f']],
              on='participant_id_canonical', how='left')
df = df.merge(path_e[['participant_id_canonical', 'cond_cluster_full', 'cond_cluster_B']],
              on='participant_id_canonical', how='left')
if survey is not None:
    df = df.merge(survey[['participant_id_canonical'] + PSYCH],
                  on='participant_id_canonical', how='left')

print(f'Merged shape: {df.shape}')
print(f'NaN in H_out_resid: {df["H_out_resid"].isna().sum()}')
print(f'Conditions: {df["study_condition"].value_counts().to_dict()}')

## Feature Correlation Gate

Blueprint rule: if |r(H_out_resid, X)| > 0.70 for any intended clustering feature X, drop X.
This prevents clustering on near-redundant information.

In [ ]:
# [3] Correlation gate — required before finalising feature set
gate_cols = ['H_out_resid', 'x_mean', 'sig_hat', 'H_emp', 'H_out', 'chi_behav']
print('=== Pearson correlation matrix ===')
corr = df[gate_cols].corr()
print(corr.round(3))
print()

GATE_THRESHOLD = 0.70
dropped = []
for f in ['x_mean', 'sig_hat', 'H_emp']:
    r, p = pearsonr(df['H_out_resid'], df[f])
    fires = abs(r) > GATE_THRESHOLD
    status = '*** GATE FIRES — DROP' if fires else 'OK'
    print(f'r(H_out_resid, {f}) = {r:.3f}, p={p:.2e}  {status}')
    if fires:
        dropped.append(f)

print()
if dropped:
    print(f'DROPPED from primary clustering: {dropped}')
    print('Replacement: H_emp (r=0.292 — acceptable, below gate threshold)')

# Final clustering features after gate
FEATURES_B = ['H_out_resid', 'sig_hat', 'H_emp']  # x_mean dropped
print(f'\nFinal primary clustering features: {FEATURES_B}')
print('x_mean → post-hoc overlay only')

## H_out_resid Descriptive Inspection

In [ ]:
# [4] H_out_resid descriptive by condition
print('=== H_out_resid stats by condition ===')
print(df.groupby('study_condition')['H_out_resid'].agg(['mean','std','min','max','count']).round(4))
print()
print('=== Path F cell × condition: H_out_resid means ===')
for cond in CONDITIONS:
    sub = df[df['study_condition'] == cond]
    print(f'  {cond}: ' + ', '.join(f'{c}={sub.loc[sub["cell_f"]==c,"H_out_resid"].mean():.3f}'
          for c in ['B_behav1','B_behav2','B_behav3','A_all']))
print()
print('Key: H_out_resid variance is lowest in default (std=0.206 vs 0.289 control, 0.300 reintervention)')

In [ ]:
# [5] Fig b1: H_out_resid distribution by condition
fig, ax = plt.subplots(figsize=(8, 5))
data_by_cond = [df.loc[df['study_condition']==c, 'H_out_resid'].values for c in CONDITIONS]
parts = ax.violinplot(data_by_cond, positions=[0,1,2], showmedians=True, showextrema=True)
for i, (pc, c) in enumerate(zip(parts['bodies'], CONDITIONS)):
    pc.set_facecolor(COND_COLORS[c]); pc.set_alpha(0.7)
parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
for i, (vals, c) in enumerate(zip(data_by_cond, CONDITIONS)):
    ax.scatter(i, np.mean(vals), color='white', edgecolor='black', s=80, zorder=5)
    ax.text(i, -0.72, f'std={np.std(vals):.3f}', ha='center', fontsize=9, color='dimgrey')
ax.set_xticks([0,1,2]); ax.set_xticklabels(CONDITIONS, fontsize=10)
ax.set_ylabel(r'$H_{out\_resid}$ (condition-residualized friction entropy)')
ax.set_title('H_out_resid by Condition\n(white dot=mean; variance lowest in default)', fontweight='bold')
ax.axhline(0, color='grey', linestyle='--', alpha=0.4)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b1_H_out_resid_by_condition.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b1')

In [ ]:
# [6] Fig b2: Correlation matrix
corr_cols = ['H_out_resid', 'x_mean', 'sig_hat', 'H_emp', 'H_out', 'chi_behav']
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols))); ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        v = corr.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(v) > 0.6 else 'black')
ax.set_title('Feature Correlation Matrix — Path B\n(gate fires: |r(H_out_resid, x_mean)|=0.707 → x_mean dropped)',
             fontweight='bold')
plt.colorbar(im, ax=ax, label='Pearson r')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b2_feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b2')

## Full-Sample Primary Analysis (N=414)

Clustering on `[H_out_resid, sig_hat, H_emp]` using full-sample StandardScaler.
Because `H_out_resid` is already condition-mean-residualized, pooling across conditions is valid.

In [ ]:
# Helper functions
def bootstrap_jaccard(X, k, n_bootstraps=N_BOOTSTRAPS, sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    n = len(X); sample_n = int(n * sample_frac)
    km_full = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=random_state)
    labels_full = km_full.fit_predict(X)
    scores = []
    for _ in range(n_bootstraps):
        idx = rng.choice(n, size=sample_n, replace=False)
        X_b = X[idx]
        km_b = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=int(rng.randint(10000)))
        labels_b = km_b.fit_predict(X_b)
        lf = labels_full[idx]; lb = labels_b
        ti, tj = np.tril_indices(sample_n, k=-1)
        sf = lf[ti] == lf[tj]; sb = lb[ti] == lb[tj]
        tp = (sf & sb).sum(); fp = (~sf & sb).sum(); fn = (sf & ~sb).sum()
        d = tp + fp + fn
        scores.append(tp / d if d > 0 else 0.0)
    return float(np.mean(scores)), float(np.std(scores))

def jaccard_flag(j):
    return 'STABLE' if j >= JACCARD_STABLE else ('MARGINAL' if j >= JACCARD_MARG else 'UNSTABLE')

def fit_ordered_km(X, k, prefix, order_col_idx=0):
    """Fit KMeans and order labels by cluster centroid value on order_col_idx ascending."""
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    raw = km.fit_predict(X)
    order = np.argsort(km.cluster_centers_[:, order_col_idx])
    lmap = {old: new for new, old in enumerate(order)}
    labels = np.array([prefix + str(lmap[l]+1) for l in raw])
    return labels, km

print('Helper functions ready.')

In [ ]:
# [7] Full-sample k-selection: k in [2, 3, 4]
sc_full = StandardScaler()
X_full  = sc_full.fit_transform(df[FEATURES_B].values)

res_full = {}
print(f'Full-sample (N={len(X_full)}):')
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    lbl = km.fit_predict(X_full)
    sizes = np.bincount(lbl); min_sz = int(sizes.min())
    if min_sz < MIN_CELL:
        print(f'  k={k}: REJECTED (min_cell={min_sz} < {MIN_CELL})')
        res_full[k] = {'feasible': False, 'min_cell': min_sz}
        continue
    sil = silhouette_score(X_full, lbl)
    ch  = calinski_harabasz_score(X_full, lbl)
    db  = davies_bouldin_score(X_full, lbl)
    j, jstd = bootstrap_jaccard(X_full, k)
    flag = jaccard_flag(j)
    feasible = j >= JACCARD_STABLE  # blueprint 5.1 step 4: selection bar is STABLE (0.75), not MARGINAL (0.60)
    print(f'  k={k}: sil={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, min_cell={min_sz}, J={j:.3f} [{flag}]')
    res_full[k] = {'sil': sil, 'ch': ch, 'db': db, 'min_cell': min_sz,
                   'jaccard': j, 'jaccard_std': jstd, 'feasible': feasible}

# Select k
K_FULL = max((k for k,r in res_full.items() if r.get('feasible')),
             key=lambda k: res_full[k]['sil'], default=None)
print(f'\n→ Selected k_full = {K_FULL} (sil={res_full[K_FULL]["sil"]:.3f}, '
      f'J={res_full[K_FULL]["jaccard"]:.3f} [{jaccard_flag(res_full[K_FULL]["jaccard"])}])')

In [ ]:
# [8] Full-sample cluster assignment (ordered by H_out_resid ascending)
labels_full, km_full = fit_ordered_km(X_full, K_FULL, 'B_fr', order_col_idx=0)
df['cluster_b_full'] = labels_full

print('=== Full-sample Path B cluster profiles ===')
profile_disp = FEATURES_B + ['x_mean', 'chi_behav', 'mu_hat']
print(df.groupby('cluster_b_full')[profile_disp].mean().round(4))
print()
print('Cluster sizes:')
print(df['cluster_b_full'].value_counts().sort_index())

In [ ]:
# [9] Fig b3: Full-sample k-selection curves
k_vals = [2, 3, 4]
sils_v = [res_full[k].get('sil', np.nan) for k in k_vals]
jacc_v = [res_full[k].get('jaccard', np.nan) for k in k_vals]
ch_v   = [res_full[k].get('ch', np.nan) for k in k_vals]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].plot(k_vals, sils_v, 'o-', color='steelblue', lw=2)
axes[0].scatter([K_FULL], [res_full[K_FULL]['sil']], s=200, color='gold', edgecolors='black', zorder=5, label=f'Selected k={K_FULL}')
axes[0].axhline(0.30, color='grey', ls='--', alpha=0.6, label='Sil=0.30')
axes[0].set(xlabel='k', ylabel='Silhouette', title='Silhouette vs k (full sample)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].plot(k_vals, ch_v, 's-', color='coral', lw=2)
axes[1].scatter([K_FULL], [res_full[K_FULL]['ch']], s=200, color='gold', edgecolors='black', zorder=5)
axes[1].set(xlabel='k', ylabel='Calinski-Harabasz', title='CH Score vs k')
axes[1].grid(alpha=0.3)
axes[2].plot(k_vals, jacc_v, '^-', color='seagreen', lw=2)
axes[2].scatter([K_FULL], [res_full[K_FULL]['jaccard']], s=200, color='gold', edgecolors='black', zorder=5)
axes[2].axhline(0.75, color='grey', ls='--', alpha=0.6, label='J=0.75 stable')
axes[2].set(xlabel='k', ylabel='Bootstrap Jaccard', title='Jaccard Stability vs k', ylim=(0, 1.05))
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
fig.suptitle(f'Full-Sample Path B k-Selection (N=414) — selected k={K_FULL}', fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b3_fullsample_cluster_selection.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b3')

In [ ]:
# [10] Fig b4: Full-sample cluster profiles
display_feats = ['H_out_resid', 'sig_hat', 'H_emp', 'x_mean']
profile_means = df.groupby('cluster_b_full')[display_feats].mean()
sizes = df['cluster_b_full'].value_counts()
clusters_full = sorted(df['cluster_b_full'].unique())
# Dynamic y-axis (bugfix): H_emp values go well above 1 (up to ~2.5), but the axis used to be
# hardcoded to (-0.4, 1.05) — bars clipped and overflow value-labels floated disconnected above
# the axes, colliding with the cluster's own "N=" title text. Compute one shared range from the
# actual data (padded) so every panel stays comparable and every label stays inside the frame.
_all_vals_b4 = profile_means[display_feats].values
_ylo_b4 = min(0.0, float(_all_vals_b4.min()))
_yhi_b4 = float(_all_vals_b4.max())
_pad_b4 = 0.12 * (_yhi_b4 - _ylo_b4 if _yhi_b4 > _ylo_b4 else 1.0)
SHARED_YLIM_B4 = (_ylo_b4 - _pad_b4 * 0.3, _yhi_b4 + _pad_b4)
fig, axes = plt.subplots(1, len(clusters_full), figsize=(13, 5))
for ax, cl in zip(axes, clusters_full):
    vals = [profile_means.loc[cl, f] for f in display_feats]
    bars = ax.bar(display_feats, vals, color=CLUSTER_COLORS_B.get(cl,'steelblue'), alpha=0.8, edgecolor='black')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{cl} (N={sizes[cl]})', fontweight='bold', color=CLUSTER_COLORS_B.get(cl,'black'))
    ax.set_ylim(*SHARED_YLIM_B4); ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(display_feats, rotation=35, ha='right', fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, max(val,0) + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
fig.suptitle(f'Full-Sample Path B Cluster Profiles (k={K_FULL}, J={res_full[K_FULL]["jaccard"]:.3f} STABLE)', fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b4_fullsample_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b4')

## Full-Sample Cross-Path Comparisons

In [ ]:
# [11] Condition distribution + ARI comparisons
ct_cond = pd.crosstab(df['cluster_b_full'], df['study_condition'])
print('=== Cluster × Condition ===')
print(ct_cond)
chi2_c, p_c, _, _ = chi2_contingency(ct_cond)
print(f'Chi2={chi2_c:.3f}, p={p_c:.4e}')
print('Row %:')
print((ct_cond.div(ct_cond.sum(axis=1), axis=0)*100).round(1))
print()

ct_p1 = pd.crosstab(df['cluster_b_full'], df['profile_p1'])
print('=== Cluster × Profile_p1 ===')
print(ct_p1)
chi2_p1, p_p1, _, _ = chi2_contingency(ct_p1)
print(f'Chi2={chi2_p1:.3f}, p={p_p1:.4e}')
print()

# ARI comparisons
ari_pa = adjusted_rand_score(df['cluster_behav'], df['cluster_b_full'])
ari_p1 = adjusted_rand_score(df['profile_p1'], df['cluster_b_full'])
cond_map = {c: i for i, c in enumerate(df['cond_cluster_full'].unique())}
ari_pe = adjusted_rand_score(df['cond_cluster_full'].map(cond_map), df['cluster_b_full'])

print('=== ARI Comparisons (full-sample) ===')
print(f'ARI(Path B vs Path A cluster_behav)  = {ari_pa:.3f}  ← {"HIGH" if ari_pa>0.5 else "moderate" if ari_pa>0.2 else "low"}')
print(f'ARI(Path B vs Phase 1 profile_p1)    = {ari_p1:.3f}  ← {"HIGH" if ari_p1>0.5 else "moderate" if ari_p1>0.2 else "low — NeT-independent"}')
print(f'ARI(Path B vs Path E cond_cluster)   = {ari_pe:.3f}')

In [ ]:
# [12] H_out_resid by Path F cell within each condition (Comparison 6)
print('=== H_out_resid by Path F cell × condition (KW tests) ===')
for cond in CONDITIONS:
    sub = df[df['study_condition'] == cond]
    groups = [sub.loc[sub['cell_f']==c, 'H_out_resid'].dropna().values
              for c in ['B_behav1','B_behav2','B_behav3','A_all']]
    stat, p = kruskal(*[g for g in groups if len(g)>1])
    print(f'  {cond}: KW H={stat:.3f}, p={p:.4e}')
    print('  Means: ' + ', '.join(f'{c}={sub.loc[sub["cell_f"]==c,"H_out_resid"].mean():.3f}'
          for c in ['B_behav1','B_behav2','B_behav3','A_all']))
print()
print('Key: B_behav1 consistently HIGHEST H_out_resid across all conditions (including default)')
print('     B_behav2 consistently LOWEST — override-heavy type has lowest friction-response diversity')

In [ ]:
# [13] Post-hoc overlays (chi_behav, mu_hat, sigma_mu_ratio, psychosocial)
print('=== chi_behav by cluster (KW) ===')
groups_chi = [df.loc[df['cluster_b_full']==c, 'chi_behav'].dropna().values
              for c in sorted(df['cluster_b_full'].unique())]
stat_chi, p_chi = kruskal(*groups_chi)
print(f'KW H={stat_chi:.3f}, p={p_chi:.4e}  *** HIGHLY SIGNIFICANT')
print(df.groupby('cluster_b_full')['chi_behav'].mean().round(2))
print('B_fr3 (diverse responder) has dramatically elevated susceptibility')
print()

if survey is not None:
    print('=== Psychosocial KW (Bonferroni n=6) ===')
    PSYCH = ['belong_mean_pre','ostracism_mean_pre','fomo_mean_pre',
             'digital_wellbeing_pre','prob_self_pre','social_media_wellbeing_pre']
    for scale in PSYCH:
        if scale not in df.columns: continue
        gs = [df.loc[df['cluster_b_full']==c, scale].dropna().values
              for c in sorted(df['cluster_b_full'].unique())]
        if all(len(g)>1 for g in gs):
            stat, p = kruskal(*gs)
            p_bonf = min(p * 6, 1.0)
            sig = '***' if p_bonf<0.001 else ('**' if p_bonf<0.01 else ('*' if p_bonf<0.05 else 'ns'))
            print(f'  {scale}: H={stat:.3f}, p_raw={p:.4e}, p_Bonf={p_bonf:.4e} {sig}')

In [ ]:
# [14] Fig b9: Path B vs Path A stacked bar
ct = pd.crosstab(df['cluster_b_full'], df['cluster_behav'])
PA_COLORS = {'C1': '#1976D2', 'C2': '#D32F2F', 'C3': '#388E3C'}
b_labels = sorted(df['cluster_b_full'].unique())
fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(b_labels))
for cl_pa, color in PA_COLORS.items():
    vals = np.array([ct.loc[b, cl_pa] if cl_pa in ct.columns and b in ct.index else 0 for b in b_labels])
    pct = vals / ct.sum(axis=1).values * 100
    ax.bar(b_labels, pct, bottom=bottom, color=color, alpha=0.8, label=f'Path A: {cl_pa}', edgecolor='white')
    for i, (v, bot) in enumerate(zip(pct, bottom)):
        if v > 4:
            ax.text(i, bot+v/2, f'{v:.0f}%', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    bottom += pct
ax.set(ylabel='% of cluster members', xlabel='Path B cluster',
       title=f'Path B vs Path A — ARI={ari_pa:.3f} (HIGH alignment)', ylim=(0, 105))
ax.legend(title='Path A cluster', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b9_pathb_vs_patha_crosspath.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b9')

In [ ]:
# [15] Fig b10: H_out_resid by Path F cell × condition
cell_order = ['B_behav1', 'B_behav2', 'B_behav3', 'A_all']
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, cond in zip(axes, CONDITIONS):
    sub = df[df['study_condition'] == cond]
    means = sub.groupby('cell_f')['H_out_resid'].mean().reindex(cell_order)
    sems  = sub.groupby('cell_f')['H_out_resid'].sem().reindex(cell_order)
    ns    = sub.groupby('cell_f').size().reindex(cell_order)
    colors = [CELL_COLORS[c] for c in cell_order]
    bars = ax.bar(cell_order, means, color=colors, alpha=0.8, edgecolor='black',
                  yerr=sems, capsize=4, error_kw={'linewidth':1.5})
    ax.axhline(0, color='black', lw=0.8, ls='--')
    groups = [sub.loc[sub['cell_f']==c, 'H_out_resid'].dropna().values for c in cell_order]
    stat, p = kruskal(*[g for g in groups if len(g)>1])
    ax.set_title(f'{cond.capitalize()}\n(KW H={stat:.1f}, p<0.0001)', fontweight='bold', color=COND_COLORS[cond])
    ax.set_xticklabels(cell_order, rotation=30, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    if ax == axes[0]: ax.set_ylabel('Mean H_out_resid')
    for bar, val, n in zip(bars, means, ns):
        if pd.notna(val):
            ax.text(bar.get_x() + bar.get_width()/2, max(val, 0) + 0.02,
                    f'N={n}', ha='center', va='bottom', fontsize=7)
fig.suptitle('H_out_resid by Path F Cell × Condition\nB_behav1=highest; B_behav2=lowest — stable pattern across ALL conditions',
             fontweight='bold', fontsize=10)
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b10_H_out_resid_by_cellf_condition.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b10')

## Within-Reintervention Analysis (PRIMARY, N=135)

Individual differences in friction-response diversity are most observable in the reintervention condition (highest H_out_resid variance: std=0.300).

In [ ]:
# [16] Within-reintervention k-selection
rei_mask = df['study_condition'] == 'reintervention'
df_rei = df[rei_mask].copy()
sc_rei = StandardScaler()
X_rei  = sc_rei.fit_transform(df_rei[FEATURES_B].values)

res_rei = {}
print(f'Reintervention (N={len(X_rei)}):')
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    lbl = km.fit_predict(X_rei)
    sizes = np.bincount(lbl); min_sz = int(sizes.min())
    if min_sz < MIN_CELL:
        print(f'  k={k}: REJECTED (min_cell={min_sz} < {MIN_CELL})')
        res_rei[k] = {'feasible': False, 'min_cell': min_sz}
        continue
    sil = silhouette_score(X_rei, lbl)
    ch  = calinski_harabasz_score(X_rei, lbl)
    db  = davies_bouldin_score(X_rei, lbl)
    j, jstd = bootstrap_jaccard(X_rei, k)
    flag = jaccard_flag(j); feasible = j >= JACCARD_STABLE  # blueprint 5.1 step 4: selection bar is STABLE (0.75), not MARGINAL (0.60)
    print(f'  k={k}: sil={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, min_cell={min_sz}, J={j:.3f} [{flag}]')
    res_rei[k] = {'sil': sil, 'ch': ch, 'db': db, 'min_cell': min_sz, 'jaccard': j, 'jaccard_std': jstd, 'feasible': feasible}

K_REI = max((k for k,r in res_rei.items() if r.get('feasible')),
            key=lambda k: res_rei[k]['sil'], default=None)
print(f'\n→ Selected k_rei = {K_REI}')
if K_REI:
    print(f'  J={res_rei[K_REI]["jaccard"]:.3f} [{jaccard_flag(res_rei[K_REI]["jaccard"])}]')

In [ ]:
# [17] Within-reintervention cluster assignment and comparisons
labels_rei, km_rei = fit_ordered_km(X_rei, K_REI, 'REI_B', order_col_idx=0)
df.loc[rei_mask, 'cluster_b_rei'] = labels_rei

print('=== Reintervention cluster profiles ===')
print(df[rei_mask].groupby('cluster_b_rei')[FEATURES_B + ['x_mean','chi_behav']].mean().round(4))
print('Sizes:', df[rei_mask]['cluster_b_rei'].value_counts().sort_index().to_dict())
print()

# ARI comparisons within reintervention
ari_rei_pa  = adjusted_rand_score(df_rei['cluster_behav'], labels_rei)
ari_rei_pe  = adjusted_rand_score(df_rei['cond_cluster_full'], labels_rei)
ari_rei_pf  = adjusted_rand_score(
    df_rei[df_rei['profile_p1']=='Profile B']['cell_f'],
    labels_rei[df_rei['profile_p1'].values == 'Profile B'])
ari_full_vs_rei = adjusted_rand_score(df[rei_mask]['cluster_b_full'], df[rei_mask]['cluster_b_rei'])

print('=== ARI comparisons — reintervention ===')
print(f'ARI(rei Path B vs Path A)              = {ari_rei_pa:.3f}')
print(f'ARI(rei Path B vs Path E rei solution) = {ari_rei_pe:.3f}  ← IDENTICAL to Path E rei!')
print(f'ARI(rei Path B vs Path F, Profile B)   = {ari_rei_pf:.3f}')
print(f'ARI(full-sample Path B vs rei Path B)  = {ari_full_vs_rei:.3f}')
print()
print('KEY FINDING: ARI(rei Path B vs Path E rei) = 1.000')
print('Path B and Path E produce IDENTICAL cluster assignments in reintervention.')
print('CONTROL: the two feature sets share [sig_hat, H_emp]; those two alone give ARI=0.941,')
print('while H_out_resid alone gives ARI=-0.001 and x_mean alone ARI=-0.005.')
print('The shared pair drives the split; neither H_out_resid nor x_mean discriminates it.')

In [ ]:
# [18] Figs b5, b6: Reintervention cluster selection and profiles
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: k-selection summary
ax = axes[0]
ax.plot([2], [res_rei.get(2, {}).get('sil', np.nan)], 'o', color='#4CAF50', markersize=14,
        label=f'k=2 selected J={res_rei.get(2,{}).get("jaccard",0):.3f} STABLE')
ax.annotate('k=3,4 REJECTED\n(min_cell<30)', xy=(3.0, 0.25), fontsize=10, color='red')
ax.axhline(0.75, color='grey', ls='--', alpha=0.6, label='J=0.75 stable')
ax.set_xlim(1.5, 4.5); ax.set_ylim(0, 1.0)
ax.set(xlabel='k', ylabel='Score', title='Reintervention k-Selection (N=135)', xticks=[2,3,4])
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Right: profile bar chart
ax = axes[1]
rei_prof = df[rei_mask].groupby('cluster_b_rei')[FEATURES_B + ['x_mean']].mean()
rei_sizes = df[rei_mask]['cluster_b_rei'].value_counts()
REI_COLORS = {'REI_B1': '#26A69A', 'REI_B2': '#EF5350'}
x = np.arange(len(FEATURES_B + ['x_mean']))
width = 0.35
display = FEATURES_B + ['x_mean']
for i_cl, (cl, col) in enumerate(REI_COLORS.items()):
    if cl not in rei_prof.index: continue
    vals = [rei_prof.loc[cl, f] for f in display]
    ax.bar(x + i_cl*width - width/2, vals, width, label=f'{cl} (N={rei_sizes.get(cl,0)})',
           color=col, alpha=0.8, edgecolor='black')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(display, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Mean value'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
ax.set_title(f'Reintervention Cluster Profiles (k={K_REI}, J={res_rei[K_REI]["jaccard"]:.3f})')
fig.suptitle('Within-Reintervention Path B (PRIMARY condition)', fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b5b6_reintervention.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b5b6')

# [18b] Fig b6 (standalone, documented filename): Reintervention cluster profiles, per-cluster
# small multiples — mirrors fig_b4's layout/bugfix. The documented output
# `fig_b6_reintervention_cluster_profiles.png` (referenced in model_run_summary_path_b.md and the
# Path B blueprint) is not produced by the combined panel above, and the combined panel's own
# right-hand axes previously showed the same hardcoded-axis overflow class of bug as fig_b4 for
# H_emp. Reusing the already-computed rei_prof/rei_sizes/REI_COLORS from the block above — no new
# statistics, just a dedicated, correctly-scaled figure saved under its documented name.
rei_clusters_b6 = [c for c in ['REI_B1', 'REI_B2'] if c in rei_prof.index]
_all_vals_b6 = rei_prof.loc[rei_clusters_b6, display].values
_ylo_b6 = min(0.0, float(_all_vals_b6.min()))
_yhi_b6 = float(_all_vals_b6.max())
_pad_b6 = 0.12 * (_yhi_b6 - _ylo_b6 if _yhi_b6 > _ylo_b6 else 1.0)
SHARED_YLIM_B6 = (_ylo_b6 - _pad_b6 * 0.3, _yhi_b6 + _pad_b6)
fig_b6, axes_b6 = plt.subplots(1, len(rei_clusters_b6), figsize=(9, 5))
for ax, cl in zip(axes_b6, rei_clusters_b6):
    vals = [rei_prof.loc[cl, f] for f in display]
    bars = ax.bar(display, vals, color=REI_COLORS.get(cl, 'steelblue'), alpha=0.8, edgecolor='black')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{cl} (N={rei_sizes[cl]})', fontweight='bold', color=REI_COLORS.get(cl, 'black'))
    ax.set_ylim(*SHARED_YLIM_B6); ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(display, rotation=35, ha='right', fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, max(val,0) + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
fig_b6.suptitle(f'Reintervention Cluster Profiles (k={K_REI}, J={res_rei[K_REI]["jaccard"]:.3f})', fontweight='bold')
plt.tight_layout()
fig_b6.savefig(FIG_PATH / 'fig_b6_reintervention_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b6 (standalone, documented filename)')

In [ ]:
# [19] Profile B within reintervention
rei_B_mask = rei_mask & (df['profile_p1'] == 'Profile B')
df_rei_B = df[rei_B_mask].copy()
print(f'Profile B within reintervention: N={len(df_rei_B)}')
sc_rei_B = StandardScaler()
X_rei_B  = sc_rei_B.fit_transform(df_rei_B[FEATURES_B].values)

res_rei_B = {}
for k in [2, 3]:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    lbl = km.fit_predict(X_rei_B)
    sizes = np.bincount(lbl); min_sz = int(sizes.min())
    if min_sz < MIN_CELL:
        print(f'  k={k}: REJECTED (min_cell={min_sz})')
        res_rei_B[k] = {'feasible': False, 'min_cell': min_sz}
        continue
    sil = silhouette_score(X_rei_B, lbl)
    j, _ = bootstrap_jaccard(X_rei_B, k)
    flag = jaccard_flag(j); feasible = j >= JACCARD_STABLE  # blueprint 5.1 step 4: selection bar is STABLE (0.75), not MARGINAL (0.60)
    print(f'  k={k}: sil={sil:.3f}, min_cell={min_sz}, J={j:.3f} [{flag}]')
    res_rei_B[k] = {'sil': sil, 'min_cell': min_sz, 'jaccard': j, 'feasible': feasible}

K_REI_B = max((k for k,r in res_rei_B.items() if r.get('feasible')),
              key=lambda k: res_rei_B[k]['sil'], default=None)
print(f'→ Selected k_rei_B = {K_REI_B}')
if K_REI_B:
    lbl_B, _ = fit_ordered_km(X_rei_B, K_REI_B, 'B_fr_rei_')
    ari_rei_B = adjusted_rand_score(df_rei_B['cell_f'], lbl_B)
    print(f'ARI(Profile B within-rei Path B vs Path F B_behav labels) = {ari_rei_B:.3f}')

## Within-Control Analysis (Counterfactual, N=137)

In [ ]:
# [20] Within-control k-selection
con_mask = df['study_condition'] == 'control'
df_con = df[con_mask].copy()
sc_con = StandardScaler()
X_con  = sc_con.fit_transform(df_con[FEATURES_B].values)

res_con = {}
print(f'Control (N={len(X_con)}):')
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    lbl = km.fit_predict(X_con)
    sizes = np.bincount(lbl); min_sz = int(sizes.min())
    if min_sz < MIN_CELL:
        print(f'  k={k}: REJECTED (min_cell={min_sz})')
        res_con[k] = {'feasible': False, 'min_cell': min_sz}
        continue
    sil = silhouette_score(X_con, lbl)
    ch  = calinski_harabasz_score(X_con, lbl)
    db  = davies_bouldin_score(X_con, lbl)
    j, jstd = bootstrap_jaccard(X_con, k)
    flag = jaccard_flag(j); feasible = j >= JACCARD_STABLE  # blueprint 5.1 step 4: selection bar is STABLE (0.75), not MARGINAL (0.60)
    print(f'  k={k}: sil={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, min_cell={min_sz}, J={j:.3f} [{flag}]')
    res_con[k] = {'sil': sil, 'ch': ch, 'db': db, 'min_cell': min_sz, 'jaccard': j, 'jaccard_std': jstd, 'feasible': feasible}

K_CON = max((k for k,r in res_con.items() if r.get('feasible')),
            key=lambda k: res_con[k]['sil'], default=None)
print(f'\n→ Selected k_con = {K_CON}')
labels_con, km_con = fit_ordered_km(X_con, K_CON, 'CON_B')
df.loc[con_mask, 'cluster_b_con'] = labels_con
print(df[con_mask].groupby('cluster_b_con')[FEATURES_B + ['x_mean','chi_behav']].mean().round(4))
ari_con_pa = adjusted_rand_score(df_con['cluster_behav'], labels_con)
ari_full_vs_con = adjusted_rand_score(df[con_mask]['cluster_b_full'], df[con_mask]['cluster_b_con'])
print(f'\nARI(control Path B vs Path A)           = {ari_con_pa:.3f}')
print(f'ARI(full-sample Path B vs control Path B) = {ari_full_vs_con:.3f}')

## Default Condition — Descriptive Only (N=142)

**No clustering attempted.** Blueprint decision, consistent with Path E Outcome D (all k rejected in default on behavioral features). H_out_resid variance is lowest in default (std=0.206 — 31% below reintervention).

In [ ]:
# [21] Default descriptive analysis
def_mask = df['study_condition'] == 'default'
df_def = df[def_mask].copy()
print('=== H_out_resid stats (default, N=142) ===')
print(df_def['H_out_resid'].describe().round(4))
print()
print('=== H_out_resid by Path F cell within default (KW) ===')
groups_def = [df_def.loc[df_def['cell_f']==c, 'H_out_resid'].dropna().values
              for c in ['B_behav1','B_behav2','B_behav3','A_all']]
stat_def, p_def = kruskal(*[g for g in groups_def if len(g)>1])
print(f'KW H={stat_def:.3f}, p={p_def:.4e}')
print(df_def.groupby('cell_f')['H_out_resid'].mean().round(4))
print()
print('FINDING: Even in default (where clustering collapses), B_behav1 shows highest H_out_resid')
print('The behavioral phenotype ordering persists on the friction-response axis.')

In [ ]:
# [22] Figs b7, b8: Control profiles and default descriptive
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: control cluster profiles
con_prof = df[con_mask].groupby('cluster_b_con')[FEATURES_B + ['x_mean']].mean()
con_sizes = df[con_mask]['cluster_b_con'].value_counts()
CON_COLORS_3 = ['#1565C0', '#42A5F5', '#90CAF9']
con_clusters = sorted(df[con_mask]['cluster_b_con'].unique())
display_feats2 = FEATURES_B + ['x_mean']
x_pos = np.arange(len(display_feats2))
for i_c, (cl, col) in enumerate(zip(con_clusters, CON_COLORS_3)):
    offset = (i_c - 1) * 0.28
    vals = [con_prof.loc[cl, f] for f in display_feats2]
    axes[0].bar(x_pos + offset, vals, 0.28, label=f'{cl} (N={con_sizes.get(cl,0)})',
                color=col, alpha=0.85, edgecolor='black')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(display_feats2, rotation=35, ha='right', fontsize=9)
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_title(f'Control k={K_CON}, J={res_con[K_CON]["jaccard"]:.3f} STABLE\n(counterfactual baseline)', fontweight='bold')

# Right: default H_out_resid by Path F cell
cell_order_d = ['B_behav1','B_behav2','B_behav3','A_all']
cell_means = df_def.groupby('cell_f')['H_out_resid'].mean().reindex(cell_order_d)
cell_sems  = df_def.groupby('cell_f')['H_out_resid'].sem().reindex(cell_order_d)
bars = axes[1].bar(cell_order_d, cell_means, color=[CELL_COLORS[c] for c in cell_order_d],
                   alpha=0.8, edgecolor='black', yerr=cell_sems, capsize=4)
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_title(f'Default H_out_resid by Path F cell\nKW H={stat_def:.1f}, p<0.0001 (NO CLUSTERING)', fontweight='bold')
axes[1].text(0.5, 0.95, 'Descriptive only\nno clustering', transform=axes[1].transAxes,
             ha='center', fontsize=9, color='red', va='top')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b7b8_control_default.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b7b8')

# [22b] Fig b7 (standalone, documented filename): Control cluster profiles, per-cluster small
# multiples — mirrors fig_b4's layout/bugfix. The documented output
# `fig_b7_control_cluster_profiles.png` (referenced in model_run_summary_path_b.md and the Path B
# blueprint) is not produced by the combined panel above. Reuses the already-computed
# con_prof/con_sizes/CON_COLORS_3/con_clusters from the block above — no new statistics, just a
# dedicated, correctly-scaled figure (dynamic y-axis instead of a hardcoded range that clipped
# H_emp) saved under its documented name.
CON_COLOR_MAP = dict(zip(con_clusters, CON_COLORS_3))
_all_vals_b7 = con_prof.loc[con_clusters, display_feats2].values
_ylo_b7 = min(0.0, float(_all_vals_b7.min()))
_yhi_b7 = float(_all_vals_b7.max())
_pad_b7 = 0.12 * (_yhi_b7 - _ylo_b7 if _yhi_b7 > _ylo_b7 else 1.0)
SHARED_YLIM_B7 = (_ylo_b7 - _pad_b7 * 0.3, _yhi_b7 + _pad_b7)
fig_b7, axes_b7 = plt.subplots(1, len(con_clusters), figsize=(13, 5))
for ax, cl in zip(axes_b7, con_clusters):
    vals = [con_prof.loc[cl, f] for f in display_feats2]
    bars = ax.bar(display_feats2, vals, color=CON_COLOR_MAP.get(cl, 'steelblue'), alpha=0.8, edgecolor='black')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{cl} (N={con_sizes[cl]})', fontweight='bold', color=CON_COLOR_MAP.get(cl, 'black'))
    ax.set_ylim(*SHARED_YLIM_B7); ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(display_feats2, rotation=35, ha='right', fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, max(val,0) + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
fig_b7.suptitle(f'Within-Control Path B Cluster Profiles (k={K_CON}, J={res_con[K_CON]["jaccard"]:.3f} STABLE)', fontweight='bold')
plt.tight_layout()
fig_b7.savefig(FIG_PATH / 'fig_b7_control_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b7 (standalone, documented filename)')

In [ ]:
# [23] Fig b11: ARI comparison heatmap
ari_rei_p1 = adjusted_rand_score(df[rei_mask]['profile_p1'], df[rei_mask]['cluster_b_rei'])
ari_con_p1 = adjusted_rand_score(df[con_mask]['profile_p1'], df[con_mask]['cluster_b_con'])
ari_con_pe = adjusted_rand_score(df[con_mask]['cond_cluster_full'], df[con_mask]['cluster_b_con'])

ari_data = {
    'vs Path A\n(behav)': {'B full-sample': round(ari_pa,3), 'B reintervention': round(ari_rei_pa,3), 'B control': round(ari_con_pa,3)},
    'vs Phase 1\n(NeT)': {'B full-sample': round(ari_p1,3), 'B reintervention': round(ari_rei_p1,3), 'B control': round(ari_con_p1,3)},
    'vs Path E\n(cond)': {'B full-sample': round(ari_pe,3), 'B reintervention': round(ari_rei_pe,3), 'B control': round(ari_con_pe,3)},
    'vs full-sample\n(Path B)': {'B full-sample': 1.000, 'B reintervention': round(ari_full_vs_rei,3), 'B control': round(ari_full_vs_con,3)},
}
ari_df = pd.DataFrame(ari_data)
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(ari_df.values, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(ari_df.columns))); ax.set_yticks(range(len(ari_df.index)))
ax.set_xticklabels(ari_df.columns, fontsize=9); ax.set_yticklabels(ari_df.index, fontsize=10)
for i in range(len(ari_df.index)):
    for j in range(len(ari_df.columns)):
        v = ari_df.values[i, j]
        ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if v > 0.8 else 'black')
plt.colorbar(im, ax=ax, label='Adjusted Rand Index')
ax.set_title('Path B ARI Comparison Matrix', fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_PATH / 'fig_b11_ari_comparison_heatmap.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved fig_b11')

## Save All Outputs

In [ ]:
# [24] Fill N/A and save cluster assignments
df['cluster_b_rei'] = df.get('cluster_b_rei', pd.Series(index=df.index)).fillna('N/A')
df['cluster_b_con'] = df.get('cluster_b_con', pd.Series(index=df.index)).fillna('N/A')

assignments = df[['participant_id_canonical', 'study_condition', 'profile_p1',
                   'cluster_b_full', 'cluster_b_rei', 'cluster_b_con',
                   'cluster_behav', 'cell_f', 'cond_cluster_full']].copy()
assignments.to_csv(PHENO_PATH / 'ds5_intervention_cluster_assignments.csv', index=False)
print(f'Saved ds5_intervention_cluster_assignments.csv ({len(assignments)} rows)')
print(assignments['cluster_b_full'].value_counts().sort_index())

In [ ]:
# [25] Save cluster profiles
profile_cols = FEATURES_B + ['x_mean', 'chi_behav', 'mu_hat']
profiles = df.groupby('cluster_b_full')[profile_cols].agg(['mean','std']).round(4)
profiles.columns = ['_'.join(c) for c in profiles.columns]
profiles['N'] = df.groupby('cluster_b_full').size()
profiles.to_csv(PHENO_PATH / 'ds5_intervention_cluster_profiles.csv')
print('Saved ds5_intervention_cluster_profiles.csv')
print(profiles)

# Save stability CSV
# Build stability records from the computed k-selection dicts (never transcribe by hand)
k_info = []
for analysis, res, n, k_sel in [
        ('full_sample', res_full, 414, K_FULL),
        ('reintervention', res_rei, 135, K_REI),
        ('profile_B_reintervention', res_rei_B, 99, K_REI_B),
        ('control', res_con, 137, K_CON)]:
    for k in sorted(res):
        r = res[k]
        rejected = 'sil' not in r
        k_info.append({
            'analysis': analysis, 'N': n, 'k_tested': k,
            'sil': None if rejected else round(r['sil'], 3),
            'min_cell': r['min_cell'],
            'jaccard': None if rejected else round(r['jaccard'], 3),
            'flag': 'REJECTED_MINCELL' if rejected else jaccard_flag(r['jaccard']),
            'selected': (k == k_sel)})
k_info.append({'analysis': 'default', 'N': 142, 'k_tested': None, 'sil': None,
               'min_cell': None, 'jaccard': None, 'flag': 'DESCRIPTIVE_ONLY', 'selected': False})
pd.DataFrame(k_info).to_csv(PHENO_PATH / 'ds5_intervention_cluster_stability.csv', index=False)
print('Saved ds5_intervention_cluster_stability.csv')

## Verification

In [ ]:
# [26] Verify all outputs
expected_files = [
    PHENO_PATH / 'ds5_intervention_cluster_assignments.csv',
    PHENO_PATH / 'ds5_intervention_cluster_assignments.md',
    PHENO_PATH / 'ds5_intervention_cluster_profiles.csv',
    PHENO_PATH / 'ds5_intervention_cluster_profiles.md',
    PHENO_PATH / 'ds5_intervention_cluster_stability.csv',
    PHENO_PATH / 'ds5_intervention_cluster_stability.md',
    PHENO_PATH / 'model_run_summary_path_b.md',
    FIG_PATH / 'fig_b1_H_out_resid_by_condition.png',
    FIG_PATH / 'fig_b2_feature_correlations.png',
    FIG_PATH / 'fig_b3_fullsample_cluster_selection.png',
    FIG_PATH / 'fig_b4_fullsample_cluster_profiles.png',
    FIG_PATH / 'fig_b9_pathb_vs_patha_crosspath.png',
    FIG_PATH / 'fig_b10_H_out_resid_by_cellf_condition.png',
    FIG_PATH / 'fig_b11_ari_comparison_heatmap.png',
]
all_ok = True
for p in expected_files:
    status = '✓' if p.exists() else '✗ MISSING'
    if not p.exists(): all_ok = False
    print(f'  {status}  {p.name}')
print()
# Verify assignment file integrity
a = pd.read_csv(PHENO_PATH / 'ds5_intervention_cluster_assignments.csv')
print(f'Assignments: {len(a)} rows, {a["cluster_b_full"].isna().sum()} NaN in cluster_b_full')
print(f'cluster_b_full values: {a["cluster_b_full"].value_counts().sort_index().to_dict()}')
print()
if all_ok:
    print('ALL OUTPUTS VERIFIED — Path B complete.')
else:
    print('WARNING: Some files missing — see above.')

## Results Summary

### Correlation gate
r(H_out_resid, x_mean) = **-0.707** → gate fires; x_mean dropped. H_emp substituted.  
Final clustering features: `[H_out_resid, sig_hat, H_emp]`.

### Full-sample (N=414)
k=3 selected (sil=0.387, J=0.940 STABLE).  
- B_fr1 (N=148): low H_out_resid (-0.211), low noise, low entropy, **high override** (0.837), moderate susceptibility (19.2)
- B_fr2 (N=198): near-zero H_out_resid, **high noise** (0.281), high entropy (2.423), low susceptibility (7.4)
- B_fr3 (N=68): **high H_out_resid** (0.467), low noise, low override (0.444), **very high susceptibility** (45.8)

**ARI(Path B vs Path A) = 0.725 — HIGH.** Path B largely recovers Path A taxonomy.  
**ARI(Path B vs Phase 1) = 0.032 — LOW.** NeT-independent.

### Within-reintervention (N=135, PRIMARY)
k=2 selected (J=0.912 STABLE). k=3,4 rejected (min_cell=19,18).  
**ARI(rei Path B vs Path E rei solution) = 1.000** — IDENTICAL cluster assignments.  
H_out_resid fully replaces x_mean as discriminator within reintervention.

### Within-control (N=137, counterfactual)
k=3 selected (J=0.808 STABLE, min_cell=30). Richer structure than reintervention.  
ARI(control Path B vs Path A) = 0.529.

### Default (N=142)
No clustering attempted. H_out_resid by Path F cell: KW H=34.3, **p<0.0001** — behavioral phenotype taxonomy predicts friction-response diversity even in the collapsed condition.

### Key finding: chi_behav
B_fr3 mean chi_behav = **45.8** (vs 19.2 and 7.4). The diverse-responder type has dramatically elevated StM friction susceptibility.

### Outcome classification
**Outcome 3 (partial):** Full-sample H_out_resid clustering largely recovers Path A taxonomy (ARI=0.725), confirming that friction-response diversity and override tendency encode nearly identical information (r=-0.707).  
**Outcome 5 (within-condition):** Within reintervention, k=2 is stable but limited to 2 clusters (k=3 infeasible). The within-condition structure is identical to Path E (ARI=1.000).

### Blueprint outcome for thesis
The DS5 behavioral phenotype taxonomy (Path A/F) is simultaneously an intervention-sensitive phenotype taxonomy. Path B adds: (1) confirmation of x_mean/H_out_resid redundancy; (2) chi_behav elevation in the diverse-responder type; (3) demonstration that Path F cell ordering on H_out_resid persists even within the default condition's collapsed behavioral space.